# Batch-consistent marker expression heatmap

This notebook uses the clustering, mapping, and marker outputs from `ResonanSC/outputs/result3` to select markers with consistent learned mapping support across batches, then plots their RNA expression across COVID-19 and healthy batches.

Batch-level `condition` labels are read from `/home/jiayu/multimodal/data/3/covid_rna.h5ad` and `/home/jiayu/multimodal/data/3/CBD-KEY-ATAC/CBD-KEY-ATAC.cell_by_peak.h5ad`. In the RNA object, `G` is treated as healthy and `H` is treated as COVID-19.

In [ ]:
from pathlib import Path
import os
import re

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="white", context="paper")
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
ROOT = Path("/home/jiayu/multimodal")
RESULT_DIR = ROOT / "ResonanSC/outputs/result3"
DATA3_DIR = ROOT / "data/3"
FIGURE_DIR = RESULT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

GENE_TOTALS = RESULT_DIR / "mapping/round_1_gene_totals.csv"
COVID_RNA_H5AD = DATA3_DIR / "covid_rna.h5ad"
COVID_ATAC_H5AD = DATA3_DIR / "CBD-KEY-ATAC/CBD-KEY-ATAC.cell_by_peak.h5ad"

TOP_N_MARKERS = 40
TOP_N_CANDIDATE_MARKERS = 250
MIN_MAPPING_BATCHES = 8
MIN_MAX_MEAN_EXPRESSION = 1e-3
PRED_COL = "pred"
LOG1P_EXPRESSION = True

OUT_PREFIX = FIGURE_DIR / "result3_data3_batch_consistent_marker_expression"

## Load batch metadata and define disease/healthy groups

In [ ]:
def rna_condition_group(condition):
    """In data/3/covid_rna.h5ad, G is healthy and H is COVID-19."""
    value = str(condition)
    if value == "G":
        return "Healthy"
    if value == "H":
        return "COVID-19"
    raise ValueError(f"Unexpected RNA condition label: {value}")

def atac_condition_group(condition):
    value = str(condition)
    return "Healthy" if value == "HV" else "COVID-19"

def batch_number(batch):
    return int(str(batch).split("_")[-1])

rna = ad.read_h5ad(COVID_RNA_H5AD, backed="r")
rna_batch_meta = (
    rna.obs[["batch", "condition", "Source", "batch_original"]]
    .drop_duplicates()
    .assign(condition_group=lambda x: x["condition"].map(rna_condition_group))
    .sort_values("batch", key=lambda s: s.map(batch_number))
    .reset_index(drop=True)
)
rna.file.close()

atac = ad.read_h5ad(COVID_ATAC_H5AD, backed="r")
atac_batch_meta = (
    atac.obs[["batch", "condition", "donor"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
atac_batch_meta.insert(0, "mapping_batch", [f"atac_{i}" for i in range(1, len(atac_batch_meta) + 1)])
atac_batch_meta = atac_batch_meta.assign(condition_group=lambda x: x["condition"].map(atac_condition_group))
atac.file.close()

batch_meta = pd.concat(
    [
        rna_batch_meta.assign(modality="RNA", mapping_batch=rna_batch_meta["batch"]),
        atac_batch_meta.assign(modality="ATAC"),
    ],
    ignore_index=True,
    sort=False,
)

display(batch_meta[["modality", "mapping_batch", "batch", "condition", "condition_group"]])

## Select resonansc markers with cross-batch mapping support

In [ ]:
gene_totals = pd.read_csv(GENE_TOTALS)
mapping_batch_cols = [c for c in gene_totals.columns if c.startswith("atac_")]

weights = gene_totals[mapping_batch_cols].fillna(0.0)
marker_scores = gene_totals[["gene"]].copy()
marker_scores["n_mapping_batches"] = (weights > 0).sum(axis=1)
marker_scores["total_mapping_weight"] = weights.sum(axis=1)
marker_scores["mean_mapping_weight"] = weights.mean(axis=1)
marker_scores["mapping_weight_cv"] = weights.std(axis=1) / weights.mean(axis=1).replace(0, np.nan)
marker_scores["consistency_score"] = (
    marker_scores["n_mapping_batches"]
    * marker_scores["mean_mapping_weight"]
    / (1.0 + marker_scores["mapping_weight_cv"].fillna(0.0))
)

adata_tmp = ad.read_h5ad(COVID_RNA_H5AD, backed="r")
rna_gene_set = set(map(str, adata_tmp.var_names))
adata_tmp.file.close()

candidate_markers = (
    marker_scores[
        marker_scores["gene"].isin(rna_gene_set)
        & (marker_scores["n_mapping_batches"] >= MIN_MAPPING_BATCHES)
    ]
    .sort_values(["n_mapping_batches", "consistency_score", "total_mapping_weight"], ascending=False)
    .head(TOP_N_CANDIDATE_MARKERS)
    .reset_index(drop=True)
)

selected_genes = candidate_markers["gene"].tolist()
display(candidate_markers.head(TOP_N_MARKERS))
print(f"Prepared {len(selected_genes)} mapping-supported candidate markers present in RNA matrices.")

## Compute marker expression per RNA batch and cluster

In [ ]:
def matrix_to_dense(x):
    if sparse.issparse(x):
        return x.toarray()
    if hasattr(x, "to_memory"):
        x = x.to_memory()
        return x.toarray() if sparse.issparse(x) else np.asarray(x)
    return np.asarray(x)

adata = ad.read_h5ad(COVID_RNA_H5AD, backed="r")
present = [g for g in selected_genes if g in adata.var_names]
missing_genes = sorted(set(selected_genes) - set(present))
gene_index = [int(adata.var_names.get_loc(g)) for g in present]

x = matrix_to_dense(adata[:, gene_index].X).astype(float)
if LOG1P_EXPRESSION:
    x = np.log1p(x)

obs = adata.obs[["batch", "condition", PRED_COL]].copy()
obs["condition_group"] = obs["condition"].map(rna_condition_group)
obs[PRED_COL] = obs[PRED_COL].astype(str)
expr_df = pd.DataFrame(x, columns=present, index=obs.index)
expr_with_obs = pd.concat([obs.reset_index(drop=True), expr_df.reset_index(drop=True)], axis=1)

batch_means = expr_with_obs.groupby(["batch", "condition", "condition_group"], observed=True)[present].mean().reset_index()
batch_sizes = expr_with_obs.groupby(["batch"], observed=True).size().rename("n_cells").reset_index()
batch_expr_long = (
    batch_means.merge(batch_sizes, on="batch", how="left")
    .melt(id_vars=["batch", "condition", "condition_group", "n_cells"], var_name="gene", value_name="mean_expression")
)

cluster_expr_long = (
    expr_with_obs.groupby(["batch", "condition", "condition_group", PRED_COL], observed=True)[present]
    .mean()
    .reset_index()
    .melt(id_vars=["batch", "condition", "condition_group", PRED_COL], var_name="gene", value_name="mean_expression")
)
adata.file.close()

batch_expr = batch_expr_long.pivot(index="gene", columns="batch", values="mean_expression").loc[selected_genes]
batch_order = (
    rna_batch_meta.assign(group_order=lambda x: x["condition_group"].map({"COVID-19": 0, "Healthy": 1}))
    .sort_values(["group_order", "batch"], key=lambda s: s if s.name != "batch" else s.str.extract(r"_(\d+)$", expand=False).astype(int))
    ["batch"]
    .tolist()
)
batch_expr = batch_expr[[b for b in batch_order if b in batch_expr.columns]]

expressed_genes = batch_expr.index[batch_expr.max(axis=1) >= MIN_MAX_MEAN_EXPRESSION].tolist()
selected_markers = (
    candidate_markers[candidate_markers["gene"].isin(expressed_genes)]
    .head(TOP_N_MARKERS)
    .reset_index(drop=True)
)
selected_genes = selected_markers["gene"].tolist()
batch_expr = batch_expr.loc[selected_genes]
cluster_expr_long = cluster_expr_long[cluster_expr_long["gene"].isin(selected_genes)].copy()

selected_markers.to_csv(f"{OUT_PREFIX}_selected_markers.csv", index=False)
batch_expr.to_csv(f"{OUT_PREFIX}_batch_mean_expression.csv")

print(
    f"Selected {len(selected_genes)} expressed markers after filtering candidates with "
    f"max batch mean expression < {MIN_MAX_MEAN_EXPRESSION}."
)
display(selected_markers)
display(batch_expr.iloc[:5, :5])

## Plot marker expression heatmap across disease and healthy RNA batches

In [ ]:
def row_zscore(df):
    centered = df.sub(df.mean(axis=1), axis=0)
    scaled = centered.div(df.std(axis=1).replace(0, np.nan), axis=0)
    return scaled.fillna(0.0)

plot_expr = row_zscore(batch_expr)
col_meta = rna_batch_meta.set_index("batch").loc[plot_expr.columns]
col_colors = col_meta["condition_group"].map({"COVID-19": "#c44e52", "Healthy": "#4c72b0"})

g = sns.clustermap(
    plot_expr,
    cmap="vlag",
    center=0,
    row_cluster=True,
    col_cluster=False,
    col_colors=col_colors,
    linewidths=0.0,
    figsize=(11, max(7, 0.20 * plot_expr.shape[0] + 3)),
    cbar_kws={"label": "Row z-score of log1p mean expression"},
)
g.ax_heatmap.set_xlabel("RNA batch")
g.ax_heatmap.set_ylabel("Batch-consistent resonansc marker")
g.ax_heatmap.set_title("resonansc markers show coherent expression across COVID-19 and healthy batches", pad=16)

for label in g.ax_heatmap.get_xticklabels():
    label.set_rotation(45)
    label.set_horizontalalignment("right")

handles = [
    plt.Line2D([0], [0], marker="s", color="w", label="COVID-19", markerfacecolor="#c44e52", markersize=9),
    plt.Line2D([0], [0], marker="s", color="w", label="Healthy", markerfacecolor="#4c72b0", markersize=9),
]
g.ax_heatmap.legend(handles=handles, title="Batch group", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False)

g.fig.savefig(f"{OUT_PREFIX}_rna_batch_heatmap.png", dpi=300, bbox_inches="tight")
g.fig.savefig(f"{OUT_PREFIX}_rna_batch_heatmap.pdf", bbox_inches="tight")
plt.show()

## Compact disease-vs-healthy summary

In [ ]:
condition_expr = (
    batch_expr.T.join(col_meta[["condition_group"]])
    .groupby("condition_group", observed=True)
    .mean()
    .T
    .reindex(selected_genes)
)
condition_expr = condition_expr[[c for c in ["COVID-19", "Healthy"] if c in condition_expr.columns]]
condition_expr.to_csv(f"{OUT_PREFIX}_condition_mean_expression.csv")

fig, ax = plt.subplots(figsize=(4.2, max(6, 0.18 * condition_expr.shape[0])))
sns.heatmap(
    row_zscore(condition_expr),
    cmap="vlag",
    center=0,
    linewidths=0.2,
    linecolor="white",
    cbar_kws={"label": "Row z-score"},
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("Batch-consistent resonansc marker")
ax.set_title("Marker expression by condition group")
fig.tight_layout()
fig.savefig(f"{OUT_PREFIX}_condition_heatmap.png", dpi=300, bbox_inches="tight")
fig.savefig(f"{OUT_PREFIX}_condition_heatmap.pdf", bbox_inches="tight")
plt.show()

## Optional: mapping support for the same selected markers

In [ ]:
mapping_weights = gene_totals.set_index("gene").loc[selected_genes, mapping_batch_cols]
atac_order = (
    atac_batch_meta.assign(group_order=lambda x: x["condition_group"].map({"COVID-19": 0, "Healthy": 1}))
    .sort_values(["group_order", "mapping_batch"], key=lambda s: s if s.name != "mapping_batch" else s.str.extract(r"_(\d+)$", expand=False).astype(int))
    ["mapping_batch"]
    .tolist()
)
mapping_weights = mapping_weights[[b for b in atac_order if b in mapping_weights.columns]]
atac_col_meta = atac_batch_meta.set_index("mapping_batch").loc[mapping_weights.columns]
atac_col_colors = atac_col_meta["condition_group"].map({"COVID-19": "#c44e52", "Healthy": "#4c72b0"})

g = sns.clustermap(
    row_zscore(mapping_weights),
    cmap="mako",
    row_cluster=True,
    col_cluster=False,
    col_colors=atac_col_colors,
    figsize=(8, max(7, 0.20 * mapping_weights.shape[0] + 3)),
    cbar_kws={"label": "Row z-score of learned mapping weight"},
)
g.ax_heatmap.set_xlabel("ATAC batch")
g.ax_heatmap.set_ylabel("Selected marker")
g.ax_heatmap.set_title("The selected markers are supported across ATAC batches", pad=16)
for label in g.ax_heatmap.get_xticklabels():
    label.set_rotation(45)
    label.set_horizontalalignment("right")
g.ax_heatmap.legend(handles=handles, title="Batch group", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False)

g.fig.savefig(f"{OUT_PREFIX}_mapping_weight_heatmap.png", dpi=300, bbox_inches="tight")
g.fig.savefig(f"{OUT_PREFIX}_mapping_weight_heatmap.pdf", bbox_inches="tight")
plt.show()